# SankhyaVox — k-NN + DTW Classifier Training

Train and evaluate the k-NN + DTW baseline on augmented data, test on held-out human speaker.

**Prerequisites:** Run `DataPipeline().build()` and augmentation first.

**Note:** DTW prediction is O(n·m) per pair and uses all training templates.
Prediction on the test set may take several minutes.

In [ ]:
# ── stdlib ──
import pickle
from pathlib import Path
from typing import Any, Dict, Iterator, List, Optional, Tuple

# ── third-party ──
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import time

## Config

In [ ]:
# ── Paths ──
DATA_PROCESSED = Path("../data_processed")
PROCESSED_DIR  = DATA_PROCESSED          # alias used by SankhyaVoxDataset default
CHECKPOINT_DIR = Path("../checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Human speaker to hold out for testing
TEST_SPEAKER = "S05"

# k-NN + DTW is slow on large datasets. Optionally limit training
# to human augments only (exclude TTS augments) for faster iteration.
USE_TTS_AUGMENTS = True  # Set False to train only on augmented human data

## SankhyaVoxDataset

**Paste the `SankhyaVoxDataset` class from `dataset/dataset.py` below.**

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  PASTE SankhyaVoxDataset class from dataset/dataset.py HERE         ║
# ╚═══════════════════════════════════════════════════════════════════════╝

## Load Data & Split

In [ ]:
# Training: augmented data, excluding held-out speaker
aug_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["augmented"])
print(f"Full augmented: {repr(aug_ds)}")
print(f"Augmented speakers: {aug_ds.speakers}")

# Exclude the test speaker's augmented variant
aug_test_speaker = f"aug{TEST_SPEAKER}"
exclude_list = [aug_test_speaker]

# Optionally exclude TTS augments for faster DTW
if not USE_TTS_AUGMENTS:
    tts_aug_speakers = [s for s in aug_ds.speakers if s.startswith("augTTS")]
    exclude_list.extend(tts_aug_speakers)
    print(f"Excluding TTS augments: {tts_aug_speakers}")

train_ds = aug_ds.exclude_speakers(exclude_list)
X_train, y_train = train_ds.get_Xy()
print(f"\nTrain: {len(X_train)} samples (excluded {exclude_list})")

# Testing: real human recordings for held-out speaker
human_ds = SankhyaVoxDataset(DATA_PROCESSED, categories=["human"])
test_ds = human_ds.filter(speaker=TEST_SPEAKER)
X_test, y_test = test_ds.get_Xy()
print(f"Test:  {len(X_test)} samples (human {TEST_SPEAKER})")
print(f"Labels in test: {sorted(set(y_test))}")
print(f"Feature shape example: {X_train[0].shape}")

---
## k-NN + DTW Classifier

**Paste the `KNNDTWClassifier` class from `models/knn_dtw_classifier.py` below.**

In [ ]:
# ╔═══════════════════════════════════════════════════════════════════════╗
# ║  PASTE KNNDTWClassifier class from models/knn_dtw_classifier.py HERE║
# ╚═══════════════════════════════════════════════════════════════════════╝

## Train

In [ ]:
knn = KNNDTWClassifier(k=3, sakoe_chiba_radius=10)
knn.fit(X_train, y_train)
print(f"Stored {len(X_train)} training templates.")

## Save

In [ ]:
knn.save(str(CHECKPOINT_DIR / "knn_dtw_classifier.pkl"))

## Load

In [ ]:
knn_loaded = KNNDTWClassifier(checkpoint_path=str(CHECKPOINT_DIR / "knn_dtw_classifier.pkl"))

## Test

In [ ]:
print(f"Running k-NN+DTW prediction on {len(X_test)} test samples...")
t0 = time.time()
knn_preds = knn_loaded.predict(X_test)
elapsed = time.time() - t0

knn_acc = accuracy_score(y_test, knn_preds)
print(f"k-NN+DTW Accuracy: {knn_acc:.3f}  ({elapsed:.1f}s)")
print()
print(classification_report(y_test, knn_preds, zero_division=0))

## Results

In [ ]:
VALUE_TO_TOKEN = {
    0: "shunya", 1: "eka", 2: "dvi", 3: "tri", 4: "catur",
    5: "pancha", 6: "shat", 7: "sapta", 8: "ashta", 9: "nava",
    10: "dasha", 20: "vimsati", 100: "shata",
}

labels_sorted = sorted(set(y_test))
label_names = [VALUE_TO_TOKEN.get(l, str(l)) for l in labels_sorted]

cm = confusion_matrix(y_test, knn_preds, labels=labels_sorted)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
            xticklabels=label_names, yticklabels=label_names)
ax.set_title(f"k-NN+DTW Confusion Matrix \u2014 Accuracy: {knn_acc:.3f}")
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
plt.tight_layout()
plt.show()

# Per-class accuracy
print("\nPer-class accuracy:")
for i, l in enumerate(labels_sorted):
    row_total = cm[i].sum()
    correct = cm[i, i]
    acc = correct / row_total if row_total > 0 else 0
    print(f"  {VALUE_TO_TOKEN.get(l, str(l)):>8s} ({l:>3d}): {correct}/{row_total} = {acc:.2%}")